# GPT-2 124M parity

Load the pinned reference weights into the scratch model and compare the fixed outputs.

In [1]:
import sys
from pathlib import Path

root = Path.cwd()
candidates = (root, root.parent,
              root / '002_GPT2_From_Scratch', root.parent / '002_GPT2_From_Scratch',
              root / '003_My GPT From Scratch', root.parent / '003_My GPT From Scratch')
package_root = next((path for path in candidates if (path / 'gpt2_scratch').is_dir()), None)
if package_root is None:
    raise FileNotFoundError('scratch package folder not found')
sys.path.insert(0, str(package_root))

import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

from gpt2_scratch import ScratchGPT2Config, ScratchGPT2ForCausalLM, load_weights_into_gpt
from gpt2_scratch.weights import REFERENCE_ALIAS, REFERENCE_REVISION

In [2]:
tokenizer = GPT2TokenizerFast.from_pretrained(REFERENCE_ALIAS, revision=REFERENCE_REVISION)
tokenizer.pad_token = tokenizer.eos_token
reference = GPT2LMHeadModel.from_pretrained(REFERENCE_ALIAS, revision=REFERENCE_REVISION, dtype=torch.float32).eval()
scratch = ScratchGPT2ForCausalLM(ScratchGPT2Config.from_scale('gpt2')).eval()
coverage = load_weights_into_gpt(scratch, reference.state_dict())

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [3]:
parameter_count = sum(parameter.numel() for parameter in scratch.parameters())
tied = scratch.out_head.weight is scratch.tok_emb.weight
print({'parameters': parameter_count, 'tied': tied, 'source_keys': len(coverage['accounted_source_keys'])})

{'parameters': 124439808, 'tied': True, 'source_keys': 149}


In [4]:
inputs = tokenizer(['Hello, I am a language model.', 'The answer is'], return_tensors='pt', padding=True)
position_ids = inputs.attention_mask.cumsum(-1) - 1
position_ids.masked_fill_(inputs.attention_mask == 0, 1)
with torch.no_grad():
    reference_logits = reference(**inputs, position_ids=position_ids).logits
    scratch_logits = scratch(**inputs, use_cache=False).logits
valid = inputs.attention_mask.bool().unsqueeze(-1).expand_as(scratch_logits)
print('max logit error:', (reference_logits[valid] - scratch_logits[valid]).abs().max().item())

max logit error: 0.0


In [5]:
prompt = tokenizer('Hello, I am a language model.', return_tensors='pt')
reference_tokens = reference.generate(**prompt, max_new_tokens=32, do_sample=False, pad_token_id=50256)
scratch_tokens = scratch.generate(**prompt, max_new_tokens=32, do_sample=False, pad_token_id=50256)
assert torch.equal(reference_tokens, scratch_tokens)
print(tokenizer.decode(scratch_tokens[0]))

Hello, I am a language model. I am a language model. I am a language model. I am a language model. I am a language model. I am a language model. I am
